In [1]:
import os
import gc
import time
import pickle
import joblib 
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def generar_shap_estratificado_cancer_rf():
    """
    Pipeline SHAP para generar análisis específicos por TIPO DE CÁNCER.
    Maneja dinámicamente la categoría base (C00_C14), filtra por umbral de robustez
    y genera salidas absolutas, porcentuales y direccionales (numéricas y categóricas).
    """
    target_name = 'MORTALIDAD'
    
    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS
    # -------------------------------------------------------------------------
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/Random_Forest"
    
    dir_base_estratificado = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}/Estratificado_Por_Cancer"
    os.makedirs(dir_base_estratificado, exist_ok=True)
    
    nombre_modelo = f"Modelo_Optimo_RF_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    # Lista completa para iterar
    categorias_cancer = [
        'CATEGORIA_CANCER_C00_C14', 'CATEGORIA_CANCER_C15_C26', 'CATEGORIA_CANCER_C30_C39', 
        'CATEGORIA_CANCER_C40_C41', 'CATEGORIA_CANCER_C43_C44', 'CATEGORIA_CANCER_C45_C49', 
        'CATEGORIA_CANCER_C50', 'CATEGORIA_CANCER_C51_C58', 'CATEGORIA_CANCER_C60_C63', 
        'CATEGORIA_CANCER_C64_C68', 'CATEGORIA_CANCER_C69_C72', 'CATEGORIA_CANCER_C73_C75', 
        'CATEGORIA_CANCER_C76_C80', 'CATEGORIA_CANCER_C81_C96', 'CATEGORIA_CANCER_C97'
    ]

    print("="*80)
    print(f"INICIANDO SHAP ESTRATIFICADO POR TIPO DE CÁNCER (TOP 10) - TARGET: {target_name}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo óptimo pre-entrenado...")
    modelo_rf = joblib.load(ruta_modelo)
        
    # Desempaquetador
    if isinstance(modelo_rf, dict):
        for k, v in modelo_rf.items():
            if hasattr(v, 'predict'): modelo_rf = v; break
    if isinstance(modelo_rf, (list, tuple)):
        for v in modelo_rf:
            if hasattr(v, 'predict'): modelo_rf = v; break
    if hasattr(modelo_rf, 'best_estimator_'): modelo_rf = modelo_rf.best_estimator_
    if hasattr(modelo_rf, 'steps'): modelo_rf = modelo_rf.steps[-1][1]
    if hasattr(modelo_rf, 'calibrated_classifiers_'): modelo_rf = modelo_rf.calibrated_classifiers_[0].estimator

    if hasattr(modelo_rf, 'feature_names_in_'):
        features = modelo_rf.feature_names_in_.tolist()
    else:
        df_dummy = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), nrows=1)
        features = [c for c in df_dummy.columns if c not in cols_excluir]

    print("-> Inicializando SHAP TreeExplainer nativo...")
    explainer = shap.TreeExplainer(modelo_rf)

    print("-> Cargando el 100% de la Cohorte Oncológica de Evaluación...")
    df_onco_completo = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

    # Identificamos las columnas OHE de cáncer que realmente existen en el dataset
    cols_cancer_reales = [col for col in df_onco_completo.columns if col.startswith('CATEGORIA_CANCER_')]

    # -------------------------------------------------------------------------
    # BUCLE PRINCIPAL POR CATEGORÍA DE CÁNCER
    # -------------------------------------------------------------------------
    for categoria in categorias_cancer:
        
        # LÓGICA ESPECIAL PARA LA CATEGORÍA BASE (C00_C14)
        if categoria == 'CATEGORIA_CANCER_C00_C14':
            mascara_base = df_onco_completo[cols_cancer_reales].sum(axis=1) == 0
            df_filtrado = df_onco_completo[mascara_base].copy()
            
        # LÓGICA NORMAL PARA LAS DEMÁS CATEGORÍAS
        else:
            if categoria not in df_onco_completo.columns:
                print(f"\nAVISO: La columna {categoria} no existe. Saltando...")
                continue
            df_filtrado = df_onco_completo[df_onco_completo[categoria] == 1].copy()
            
        n_pacientes = len(df_filtrado)
        nombre_limpio = categoria.replace('CATEGORIA_CANCER_', '')
        UMBRAL_MINIMO = 100
        
        if n_pacientes < UMBRAL_MINIMO:
            print(f"\nAVISO: La categoría {nombre_limpio} no supera el umbral mínimo de {UMBRAL_MINIMO} personas. Excluyendo por falta de robustez estadística.")
            continue
        
        print("\n" + "-"*60)
        print(f"--- PROCESANDO SUBGRUPO: {nombre_limpio} ({n_pacientes} pacientes) ---")
        print("-"*60)
            
        dir_sub_enfoque = os.path.join(dir_base_estratificado, nombre_limpio)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        X_shap = df_filtrado[features].astype('float32')
        del df_filtrado; gc.collect()
        
        inicio_time = time.time()
        
        batch_size = 500
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            if (i // batch_size + 1) % 10 == 0 or (i // batch_size + 1) == 1:
                print(f"      -> Bloque {i//batch_size + 1} de {n_batches}...")
            
            shap_output = explainer.shap_values(batch, check_additivity=False, approximate=True)
            
            if isinstance(shap_output, list):
                shap_mat_batch = np.stack(shap_output, axis=2)
            elif len(shap_output.shape) == 3:
                shap_mat_batch = shap_output
            else:
                shap_mat_batch = np.stack([shap_output * -1, shap_output], axis=2)
                
            resultados_list.append(shap_mat_batch)
            del batch, shap_output, shap_mat_batch; gc.collect()
            
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP calculado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO AUTOMÁTICO DE CONSTANTES ---
        varianzas = X_shap.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        
        if cols_a_eliminar:
            idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
            X_shap = X_shap.drop(columns=cols_a_eliminar)
            matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
            print(f"   -> Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        n_clases = matriz_shap.shape[2]
        
        ruta_npy = os.path.join(dir_sub_enfoque, f"MATRIZ_SHAP_{nombre_limpio}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # Exportar CSV de Importancias (Absoluto)
        shap_abs = np.abs(matriz_shap).mean(axis=0)
        impacto_total = shap_abs.sum(axis=1)
        
        columnas_csv = [f"Clase_{i}_" + ("Vivo" if i==0 else "Fallecido") for i in range(n_clases)]
        df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
        df_shap_imp['Impacto_Total'] = impacto_total
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')
        
        # CSV en Porcentajes
        print("   -> Generando CSV de importancias en porcentajes...")
        df_shap_porcentajes = (df_shap_imp / df_shap_imp.sum()) * 100
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Gráfico Summary General (Top 10)
        plt.figure(figsize=(10, 6))
        df_top10 = df_shap_imp.head(10).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top10.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='bwr', ax=plt.gca()) 
        plt.title(f'Top 10 Variables SHAP - Cáncer {nombre_limpio} (Mortalidad)', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Rangos Numéricos Absolutos
        print("   -> Calculando impacto absoluto por rangos numéricos...")
        rangos_resumen = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                for idx_c, rango in enumerate(bins_serie.cat.categories):
                    indices_rango = (bins_serie == rango)
                    if indices_rango.sum() > 0:
                        impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num), :]).mean(axis=0)
                        dict_rango = {"Variable": v_num, "Rango": str(rango), "N_Pacientes": indices_rango.sum()}
                        for cl in range(n_clases):
                            etiqueta = "Vivo" if cl == 0 else "Fallecido"
                            dict_rango[f"Impacto_Promedio_Clase_{cl}_{etiqueta}"] = impacto_medio_rango[cl]
                        rangos_resumen.append(dict_rango)
                            
        df_rangos = pd.DataFrame(rangos_resumen)
        df_rangos.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Impacto_Variables_Numericas_{nombre_limpio}.csv"), index=False)
        
        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL: VARIABLES NUMÉRICAS
        # -------------------------------------------------------------------------
        print("   -> Calculando impacto direccional (Mortalidad) por rangos numéricos...")
        rangos_direccionales = []
        matriz_fallecido = matriz_shap[:, :, 1]
        
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes_rango = indices_rango.sum()
                    
                    if n_pacientes_rango > 0:
                        valores_crudos = matriz_fallecido[indices_rango, idx_var]
                        promedio_crudo = valores_crudos.mean()
                        
                        if promedio_crudo > 0:
                            efecto = "Aumenta Mortalidad (+)"
                        elif promedio_crudo < 0:
                            efecto = "Protector / Supervivencia (-)"
                        else:
                            efecto = "Neutral"

                        rangos_direccionales.append({
                            "Variable": v_num,
                            "Rango": str(rango),
                            "N_Pacientes": n_pacientes_rango,
                            "SHAP_Promedio_Crudo_Mortalidad": promedio_crudo,
                            "Efecto_Clinico": efecto
                        })
                        
        df_direccional = pd.DataFrame(rangos_direccionales)
        df_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccionales_Mortalidad_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # ANÁLISIS DIRECCIONAL: VARIABLES CATEGÓRICAS (OHE)
        # -------------------------------------------------------------------------
        print("   -> Calculando impacto direccional (Mortalidad) para variables categóricas (OHE)...")
        vars_cat_ohe = [col for col in X_shap.columns if col not in vars_num]
        cat_direccionales = []
        
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_fallecido[indices_cat, idx_var].mean()
                    
                    if promedio_crudo > 0:
                        efecto = "Aumenta Mortalidad (+)"
                    elif promedio_crudo < 0:
                        efecto = "Protector / Supervivencia (-)"
                    else:
                        efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        "SHAP_Promedio_Crudo_Mortalidad": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_Mortalidad_{nombre_limpio}.csv"), index=False)

        # -------------------------------------------------------------------------
        # PANELES DE DEPENDENCIA (TOP 10)
        # -------------------------------------------------------------------------
        print(f"   -> Generando paneles de dependencia para el Top 10...")
        top_10_vars = df_shap_imp.head(10).index.tolist()
        
        for var in top_10_vars:
            if var in X_shap.columns:
                fig, ax = plt.subplots(figsize=(6, 4.5))
                valores_sh_fallecido = matriz_shap[:, :, 1]
                
                shap.dependence_plot(
                    var, valores_sh_fallecido, X_shap, 
                    interaction_index=None, ax=ax, show=False
                )
                ax.set_title(f'Impacto en Riesgo de Fallecimiento (Clase 1)', fontsize=10)
                fig.suptitle(f'Dependence Plot: {var} ({nombre_limpio})', fontsize=11, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes, df_direccional, df_cat_direccional; gc.collect()

    print("\n" + "="*80)
    print("PROCESO DE ESTRATIFICACIÓN CONCLUIDO CON ÉXITO")
    print(f"Resultados guardados en:\n{dir_base_estratificado}")
    print("="*80)
    
    del df_onco_completo; gc.collect()

# Ejecutar el script estratificado
generar_shap_estratificado_cancer_rf()

INICIANDO SHAP ESTRATIFICADO POR TIPO DE CÁNCER (TOP 10) - TARGET: MORTALIDAD
Hora de inicio: 2026-07-16 06:25:33
-> Cargando modelo óptimo pre-entrenado...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

AVISO: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Bloque 1 de 53...
      -> Bloque 10 de 53...
      -> Bloque 20 de 53...
      -> Bloque 30 de 53...
      -> Bloque 40 de 53...
      -> Bloque 50 de 53...
   -> SHAP calculado en 0.48 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Calculando impacto direccional (Mortalidad) por rangos numéricos...
  

In [1]:
import os
import gc
import time
import pickle
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

def generar_shap_estratificado_cancer_xgb(target_name):
    """
    Pipeline SHAP para generar análisis estratificados por TIPO DE CÁNCER 
    para modelos XGBoost Multiclase (SEVERIDAD y CONSUMO_RECURSOS).
    Filtra pacientes (< 100), maneja la clase base C00_C14 y extrae rangos 
    direccionales (numéricos y categóricos) de la clase más alta.
    """
    # Configuración dinámica del índice de mayor riesgo según el target
    if target_name == 'SEVERIDAD':
        idx_clase_alta = 3  # Clase 3 corresponde al índice 3 (0, 1, 2, 3)
        nombre_efecto_str = 'Severidad Alta (Clase 3)'
    else:
        idx_clase_alta = 2  # Clase 2 corresponde al índice 2 (0, 1, 2)
        nombre_efecto_str = 'Consumo Alto (Clase 2)'

    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS
    # -------------------------------------------------------------------------
    dir_datos = "../../Datos/Datasets Finales"
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/XGBoost"
    
    dir_base_estratificado = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}/Estratificado_Por_Cancer"
    os.makedirs(dir_base_estratificado, exist_ok=True)
    
    nombre_modelo = f"Modelo_Optimo_XGBoost_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    categorias_cancer = [
        'CATEGORIA_CANCER_C00_C14', 'CATEGORIA_CANCER_C15_C26', 'CATEGORIA_CANCER_C30_C39',
        'CATEGORIA_CANCER_C40_C41', 'CATEGORIA_CANCER_C43_C44', 'CATEGORIA_CANCER_C45_C49',
        'CATEGORIA_CANCER_C50', 'CATEGORIA_CANCER_C51_C58', 'CATEGORIA_CANCER_C60_C63',
        'CATEGORIA_CANCER_C64_C68', 'CATEGORIA_CANCER_C69_C72', 'CATEGORIA_CANCER_C73_C75',
        'CATEGORIA_CANCER_C76_C80', 'CATEGORIA_CANCER_C81_C96', 'CATEGORIA_CANCER_C97'
    ]

    print("="*80)
    print(f"INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: {target_name} (XGBOOST)")
    print(f"Enfocando análisis direccional en: {nombre_efecto_str}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo óptimo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo XGBoost pre-entrenado desde: {nombre_modelo}...")
    with open(ruta_modelo, 'rb') as f:
        modelo_xgb = pickle.load(f)
        
    print("-> Inicializando SHAP TreeExplainer nativo...")
    explainer = shap.TreeExplainer(modelo_xgb)
    features = modelo_xgb.get_booster().feature_names

    print("-> Cargando el 100% de la Cohorte Oncológica de Evaluación...")
    df_onco_completo = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)

    cols_cancer_reales = [col for col in df_onco_completo.columns if col.startswith('CATEGORIA_CANCER_')]

    # -------------------------------------------------------------------------
    # BUCLE PRINCIPAL POR CATEGORÍA DE CÁNCER
    # -------------------------------------------------------------------------
    for categoria in categorias_cancer:
        
        # LÓGICA ESPECIAL PARA LA CATEGORÍA BASE (C00_C14)
        if categoria == 'CATEGORIA_CANCER_C00_C14':
            mascara_base = df_onco_completo[cols_cancer_reales].sum(axis=1) == 0
            df_filtrado = df_onco_completo[mascara_base].copy()
            
        # LÓGICA NORMAL PARA LAS DEMÁS CATEGORÍAS
        else:
            if categoria not in df_onco_completo.columns:
                print(f"\nADVERTENCIA: La columna {categoria} no existe. Saltando...")
                continue
            df_filtrado = df_onco_completo[df_onco_completo[categoria] == 1].copy()
            
        n_pacientes = len(df_filtrado)
        nombre_limpio = categoria.replace('CATEGORIA_CANCER_', '')
        UMBRAL_MINIMO = 100
        
        if n_pacientes < UMBRAL_MINIMO:
            print(f"\nADVERTENCIA: La categoría {nombre_limpio} no supera el umbral mínimo de {UMBRAL_MINIMO} personas. Excluyendo por falta de robustez estadística.")
            continue
            
        print("\n" + "-"*60)
        print(f"--- PROCESANDO SUBGRUPO: {nombre_limpio} ({n_pacientes} pacientes) ---")
        print("-"*60)
        
        dir_sub_enfoque = os.path.join(dir_base_estratificado, nombre_limpio)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        X_shap = df_filtrado[features].astype('float32')
        del df_filtrado; gc.collect()
        
        inicio_time = time.time()
        
        batch_size = 10000
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        for i in range(0, len(X_shap), batch_size):
            batch = X_shap.iloc[i:i+batch_size]
            print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            
            shap_obj = explainer(batch)
            resultados_list.append(shap_obj.values)
            del batch, shap_obj; gc.collect()
            
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO AUTOMÁTICO DE CONSTANTES ---
        varianzas = X_shap.var()
        cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
        
        if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
            cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
            
        if cols_a_eliminar:
            idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
            X_shap = X_shap.drop(columns=cols_a_eliminar)
            
            if len(matriz_shap.shape) == 3: 
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
            else:
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                
            print(f"   -> Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        n_clases = matriz_shap.shape[2] if len(matriz_shap.shape) == 3 else 1
        
        # Guardar matriz .npy
        ruta_npy = os.path.join(dir_sub_enfoque, f"MATRIZ_SHAP_{nombre_limpio}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # Exportar CSV de Importancias Agregadas
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        if n_clases > 1:
            impacto_total = shap_abs.sum(axis=1)
            columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
            df_shap_imp['Impacto_Total'] = impacto_total
        else:
            df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=['Impacto_Total'])
            
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')
        
        # CSV en Porcentajes
        print("   -> Generando CSV de importancias en porcentajes...")
        df_shap_porcentajes = (df_shap_imp / df_shap_imp.sum()) * 100
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{nombre_limpio}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Gráfico Summary General Stacked (TOP 10)
        plt.figure(figsize=(10, 6))
        df_top10 = df_shap_imp.head(10)
        if n_clases > 1:
            df_top10 = df_top10.drop(columns=['Impacto_Total']).iloc[::-1]
            df_top10.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='viridis', ax=plt.gca())
        else:
            df_top10['Impacto_Total'].iloc[::-1].plot(kind='barh', figsize=(10, 6), color='teal', ax=plt.gca())
            
        plt.title(f'Top 10 Variables SHAP - Cáncer {nombre_limpio} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Gráfico Summary Plot Categóricas (TOP 10)
        plt.figure(figsize=(10, 6))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top10_cat = df_shap_imp.loc[vars_cat_ohe].head(10)
        if n_clases > 1:
            df_top10_cat = df_top10_cat.drop(columns=['Impacto_Total']).iloc[::-1]
            df_top10_cat.plot(kind='barh', stacked=True, figsize=(10, 6), cmap='plasma', ax=plt.gca())
        else:
            df_top10_cat['Impacto_Total'].iloc[::-1].plot(kind='barh', figsize=(10, 6), color='purple', ax=plt.gca())
            
        plt.title(f'Top 10 Categóricas SHAP - Cáncer {nombre_limpio} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{nombre_limpio}.png"), dpi=300)
        plt.close()
        
        # Análisis cuantitativo absoluto por rangos
        print("   -> Calculando impacto absoluto por rangos numéricos...")
        rangos_resumen = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                for idx_c, rango in enumerate(bins_serie.cat.categories):
                    indices_rango = (bins_serie == rango)
                    if indices_rango.sum() > 0:
                        if n_clases > 1:
                            impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num), :]).mean(axis=0)
                            dict_rango = {"Variable": v_num, "Rango": str(rango), "N_Pacientes": indices_rango.sum()}
                            for cl in range(n_clases):
                                dict_rango[f"Impacto_Promedio_Clase_{cl}"] = impacto_medio_rango[cl]
                            rangos_resumen.append(dict_rango)
                        else:
                            impacto_medio_rango = np.abs(matriz_shap[indices_rango, X_shap.columns.get_loc(v_num)]).mean()
                            rangos_resumen.append({
                                "Variable": v_num, "Rango": str(rango), 
                                "N_Pacientes": indices_rango.sum(), "Impacto_Promedio": impacto_medio_rango
                            })
                            
        df_rangos = pd.DataFrame(rangos_resumen)
        df_rangos.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Impacto_Variables_Numericas_{nombre_limpio}.csv"), index=False)
        
        # Análisis Direccional Crudo NUMÉRICO (Para la clase de mayor riesgo)
        print(f"   -> Calculando impacto direccional ({nombre_efecto_str}) por rangos numéricos...")
        rangos_direccionales = []
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta] if n_clases > 1 else matriz_shap
        
        for v_num in vars_num:
            if v_num in X_shap.columns:
                try:
                    bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except:
                    bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                idx_var = X_shap.columns.get_loc(v_num)
                
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes_rango = indices_rango.sum()
                    
                    if n_pacientes_rango > 0:
                        valores_crudos = matriz_clase_alta[indices_rango, idx_var]
                        promedio_crudo = valores_crudos.mean()
                        
                        if promedio_crudo > 0:
                            efecto = "Aumenta probabilidad (+)"
                        elif promedio_crudo < 0:
                            efecto = "Disminuye probabilidad (-)"
                        else:
                            efecto = "Neutral"

                        rangos_direccionales.append({
                            "Variable": v_num,
                            "Rango": str(rango),
                            "N_Pacientes": n_pacientes_rango,
                            f"SHAP_Promedio_Crudo_{target_name}": promedio_crudo,
                            "Efecto_Clinico": efecto
                        })
                        
        df_direccional = pd.DataFrame(rangos_direccionales)
        df_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccionales_{target_name}_{nombre_limpio}.csv"), index=False)

        # --- NUEVO: Análisis Direccional Crudo CATEGÓRICO (OHE 0 vs 1) ---
        print(f"   -> Calculando impacto direccional ({nombre_efecto_str}) para variables categóricas (OHE)...")
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    
                    if promedio_crudo > 0:
                        efecto = "Aumenta probabilidad (+)"
                    elif promedio_crudo < 0:
                        efecto = "Disminuye probabilidad (-)"
                    else:
                        efecto = "Neutral"

                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Promedio_Crudo_{target_name}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{nombre_limpio}.csv"), index=False)


        # Paneles de Dependencia Multiclase (Centrado en el Top 10)
        print("   -> Generando paneles de dependencia para el Top 10...")
        top_10_vars = df_shap_imp.head(10).index.tolist()
        
        for var in top_10_vars:
            if var in X_shap.columns:
                fig, axes = plt.subplots(1, n_clases, figsize=(5 * n_clases, 4.5))
                if n_clases == 1: axes = [axes]
                
                for clase in range(n_clases):
                    valores_sh_clase = matriz_shap[:, :, clase] if n_clases > 1 else matriz_shap
                    shap.dependence_plot(
                        var, valores_sh_clase, X_shap, 
                        interaction_index=None, ax=axes[clase], show=False
                    )
                    axes[clase].set_title(f'Impacto en clase {clase}', fontsize=10)
                
                fig.suptitle(f'Dependence Plot: {var} ({nombre_limpio})', fontsize=12, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes, df_direccional, df_cat_direccional; gc.collect()

    print("\n" + "="*80)
    print("PROCESO DE ESTRATIFICACIÓN CONCLUIDO CON ÉXITO")
    print(f"Resultados guardados en subcarpetas dentro de:\n{dir_base_estratificado}")
    print("="*80)
    
    del df_onco_completo; gc.collect()

# Ejemplo de ejecución:
# generar_shap_estratificado_cancer_xgb('SEVERIDAD')
# generar_shap_estratificado_cancer_xgb('CONSUMO_RECURSOS')

In [3]:
# Ejecutar el script para ambos targets
generar_shap_estratificado_cancer_xgb('SEVERIDAD')

INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: SEVERIDAD (XGBOOST)
Enfocando análisis direccional en: Severidad Alta (Clase 3)
Hora de inicio: 2026-07-15 04:42:07
-> Cargando modelo XGBoost pre-entrenado desde: Modelo_Optimo_XGBoost_SEVERIDAD.pkl...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

ADVERTENCIA: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Procesando bloque 1 de 3...
      -> Procesando bloque 2 de 3...
      -> Procesando bloque 3 de 3...
   -> SHAP completado en 1.53 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Calculando imp

In [2]:
generar_shap_estratificado_cancer_xgb('CONSUMO_RECURSOS')

INICIANDO SHAP ESTRATIFICADO POR CÁNCER (TOP 10) - TARGET: CONSUMO_RECURSOS (XGBOOST)
Enfocando análisis direccional en: Consumo Alto (Clase 2)
Hora de inicio: 2026-07-15 04:02:25
-> Cargando modelo XGBoost pre-entrenado desde: Modelo_Optimo_XGBoost_CONSUMO_RECURSOS.pkl...
-> Inicializando SHAP TreeExplainer nativo...
-> Cargando el 100% de la Cohorte Oncológica de Evaluación...

ADVERTENCIA: La categoría C00_C14 no supera el umbral mínimo de 100 personas. Excluyendo por falta de robustez estadística.

------------------------------------------------------------
--- PROCESANDO SUBGRUPO: C15_C26 (26320 pacientes) ---
------------------------------------------------------------
      -> Procesando bloque 1 de 3...
      -> Procesando bloque 2 de 3...
      -> Procesando bloque 3 de 3...
   -> SHAP completado en 1.31 minutos.
   -> Se excluyeron 17 variables constantes.
   -> Generando CSV de importancias en porcentajes...
   -> Calculando impacto absoluto por rangos numéricos...
   -> Ca